# Combine and process Cleanalyze results

Author: **Niels J. de Winter** (*n.j.de.winter@vu.nl*)<br>
Assistant Professor Vrije Universiteit Amsterdam

## Load packages

In [69]:
import os
import pandas as pd

## Read all cleanalyze data

In [70]:
folder_path = "TE_data_with_growth_rates"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')] # List all CSV files in the folder
dataframes = {}

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_csv(file_path)
    # round timing info according to different peak IDs to nearest second
    df['timing_1'] = pd.to_datetime(df['timing_1'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df['timing_2'] = pd.to_datetime(df['timing_2'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df['timing_3'] = pd.to_datetime(df['timing_3'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df['timing_4'] = pd.to_datetime(df['timing_4'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    dataframes[file] = df

# Print the first few rows of each DataFrame
for file, df in dataframes.items():
    print(f"Data from {file}:")
    print(df.head(), "\n")  # Display the first few rows of each DataFrame
    print(f"Shape: {df.shape}\n")  # Print the shape of the DataFrame
    print(f"Columns: {df.columns.tolist()}\n")  # Print the column names
    print("-" * 40)  # Separator for clarity

Data from B255_int_peakid.csv:
      depth  proxy_filtered      time         Xpos         Ypos         Ca43  \
0  0.445779        0.013676  62.33211  30944.10620  80703.27530  6301.588000   
1  0.891558        0.000000  62.44125  30944.46978  80703.01737  5301.123838   
2  1.337336        0.019326  62.55040  30944.83337  80702.75945  4600.846556   
3  1.783115        0.037763  62.65955  30945.19695  80702.50152  5751.322804   
4  2.228894        0.013795  62.76871  30945.56053  80702.24359  6251.562891   

    23Na/43Ca  25Mg/43Ca  43Ca/43Ca  55Mn/43Ca  ...  \
0  199.059248   1.253571       1000   0.306491  ...   
1  235.482235   3.406310       1000   0.347474  ...   
2  284.331679   1.771388       1000   0.380053  ...   
3  221.307763   3.115197       1000   0.348871  ...   
4  205.558351   1.264443       1000   0.242893  ...   

   interval_growth_rate_peakID_3  interval_growth_rate_centered_peakID_3  \
0                      20.329382                               20.329382   
1    

## Load temperature and salinity data

In [71]:
# Load the environmental data
env_data_path = "min60_2022.csv"
env_df = pd.read_csv(env_data_path, sep=None, engine='python')

# Remove spaces from column names
env_df.columns = env_df.columns.str.replace(' ', '', regex=True)

# Parse the date column (TM)
env_df['TM'] = pd.to_datetime(env_df['TM'], format='%Y%m%d%H%M%S', errors='coerce')
print(env_df.head())

                   TM           ET      T  T_std T_N  T_max  T_min  T_flag  \
0 2022-01-01 01:00:00  44562.04167  6.562  0.105   6  6.677  6.438     1.0   
1 2022-01-01 02:00:00  44562.08333  6.676  0.059   6  6.805  6.646     1.0   
2 2022-01-01 03:00:00  44562.12500  6.982   0.05   6  7.031  6.906     1.0   
3 2022-01-01 04:00:00  44562.16667  7.008  0.073   6  7.183  6.991     1.0   
4 2022-01-01 05:00:00  44562.20833  7.141  0.093   6  7.244  7.024     1.0   

        S  S_std S_N   S_max   S_min  S_flag  
0  28.145  0.288   6  28.435  27.795     1.0  
1  28.465  0.224   6  29.005  28.415     1.0  
2  29.395  0.121   6  29.505  29.225     1.0  
3  29.385  0.208   6  29.875  29.335     1.0  
4   29.67  0.305   6  30.065  29.315     1.0  


## Load high tide series data

In [72]:
# Load the data from high tide measurements
HW_data_path = "HWseries_2022.csv"
HW_df = pd.read_csv(HW_data_path, sep=None, engine='python')

# Convert the date column (Timestamp [UTC]) to datetime format
HW_df['TM'] = pd.to_datetime(HW_df['Timestamp [UTC]'], format='%d/%m/%Y %H:%M', errors='coerce')
print(HW_df.head())

  SampleID   Timestamp [UTC]    T     S SecchiDepth      TN     TP    PO4  \
0   HW2201  04/01/2022 09:06  8.0  30.8         0.8  40.756  0.980  0.811   
1   HW2202  12/01/2022 13:04  6.0  26.6         1.2  53.431  0.794  0.628   
2   HW2203  02/02/2022 09:05  6.6  28.0         0.9  45.559  0.638  0.448   
3   HW2204  24/02/2022 10:27  7.0  28.7         0.7  61.718  0.789  0.642   
4   HW2205  10/03/2022 10:04  6.1  24.1         0.7  91.492  0.474  0.280   

      NO3    NO2    NH4     DON    DOP     Si     DIC    TSM     Chl  \
0  24.253  1.078  6.940   8.485  0.169  18.01  2319.5  24.20   1.715   
1  29.230  1.560  8.917  13.724  0.166  21.57  2371.0  13.40   1.651   
2  26.868  1.111  5.724  11.856  0.190  16.71  2335.8  17.65   2.087   
3  47.019  0.950  4.772   8.977  0.147  27.46  2301.2  39.60   1.451   
4  67.368  0.984  4.682  18.458  0.194  34.73  2396.9  36.30  12.143   

                   TM  
0 2022-01-04 09:06:00  
1 2022-01-12 13:04:00  
2 2022-02-02 09:05:00  
3 2022-0

### Combine environmental data with TE data

In [74]:
# For all dataframes in 'dataframes', merge with env_df and add only T, T_std, S, S_std columns
for fname, df in dataframes.items():
    # Merge and select only required columns from env_df for each timing option
    for timing_col in ['timing_1', 'timing_2', 'timing_3', 'timing_4']:
        # Merge with env_df to get temperature and salinity data aligned to the specific timing column
        merged = pd.merge_asof(
            df.sort_values(timing_col),
            env_df[['TM', 'T', 'T_std', 'S', 'S_std']].sort_values('TM'),
            left_on=timing_col,
            right_on='TM',
            direction='nearest'
        )
        # Merge with HW_df to get high tide data
        merged = pd.merge_asof(
            merged.sort_values(timing_col),
            HW_df[['TM', 'Chl']].sort_values('TM'),
            left_on=timing_col,
            right_on='TM',
            direction='nearest'
        )

        # Add/overwrite columns in the original dataframe
        dataframes[fname][f'T_{timing_col}'] = merged['T'].values
        dataframes[fname][f'T_std_{timing_col}'] = merged['T_std'].values
        dataframes[fname][f'S_{timing_col}'] = merged['S'].values
        dataframes[fname][f'S_std_{timing_col}'] = merged['S_std'].values
        dataframes[fname][f'Chl_{timing_col}'] = merged['Chl'].values

    # Print the updated DataFrame head
    print(f"Updated data from {fname}:")
    print(dataframes[fname].head(), "\n")  # Display the first few rows of the updated DataFrame

    # Save the updated DataFrame in a new folder
    output_folder = "TE_data_with_GR_T_S"
    os.makedirs(output_folder, exist_ok=True)  # Create the folder if it doesn't exist
    output_file_path = os.path.join(output_folder, fname)
    dataframes[fname].to_csv(output_file_path, index=False)  # Save the updated DataFrame to CSV


Updated data from B255_int_peakid.csv:
      depth  proxy_filtered      time         Xpos         Ypos         Ca43  \
0  0.445779        0.013676  62.33211  30944.10620  80703.27530  6301.588000   
1  0.891558        0.000000  62.44125  30944.46978  80703.01737  5301.123838   
2  1.337336        0.019326  62.55040  30944.83337  80702.75945  4600.846556   
3  1.783115        0.037763  62.65955  30945.19695  80702.50152  5751.322804   
4  2.228894        0.013795  62.76871  30945.56053  80702.24359  6251.562891   

    23Na/43Ca  25Mg/43Ca  43Ca/43Ca  55Mn/43Ca  ...  T_timing_3  \
0  199.059248   1.253571       1000   0.306491  ...       5.843   
1  235.482235   3.406310       1000   0.347474  ...       5.843   
2  284.331679   1.771388       1000   0.380053  ...       5.843   
3  221.307763   3.115197       1000   0.348871  ...       5.843   
4  205.558351   1.264443       1000   0.242893  ...       5.843   

   T_std_timing_3 S_timing_3 S_std_timing_3 Chl_timing_3 T_timing_4  \
0     